### 1: Imports

In [1]:
import pandas as pd
import numpy as np
import pickle
import wandb
from pathlib import Path

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, f1_score

In [3]:
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [4]:
OUTPUT_DIR = Path('../outputs')
DATA_DIR   = Path('../data')

###   2:   MAP@3 + HELPERS FUNCTION

In [5]:
ANSWER_MAP  = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
REVERSE_MAP = {v: k for k, v in ANSWER_MAP.items()}

In [6]:
"""
MAP@3: pos 1→1.0, 
       pos 2→0.5, 
       pos 3→0.33,
       miss→0.0
"""
def map_at_3(y_true, y_proba):
    scores = []
    for i, true in enumerate(y_true):
        top3 = np.argsort(y_proba[i])[-3:][::-1]
        score = (1.0 / (np.where(top3 == true)[0][0] + 1) if true in top3 else 0.0) 
        scores.append(score)
    return float(np.mean(scores))

In [7]:
def make_submission(model, vectorizer, test_df, test_ids):
    X = vectorizer.transform(test_df['combined_text'].values)
    proba = model.predict_proba(X)
    preds = []
    for i in range(len(test_df)):
        top3 = np.argsort(proba[i])[-3:][::-1]
        preds.append(' '.join([REVERSE_MAP[j] for j in top3]))
    return pd.DataFrame({'ID': test_ids, 
                         'Prediction': preds})

In [9]:
def evaluate(model, vectorizer, df, split_name='val'):
    X     = vectorizer.transform(df['combined_text'].values)
    y     = df['answer'].map(ANSWER_MAP).values
    pred  = model.predict(X)
    proba = model.predict_proba(X)
    acc   = accuracy_score(y, pred)
    f1    = f1_score(y, pred, average='weighted')
    m3    = map_at_3(y, proba)
    print(f'  [{split_name}] ACC={acc:.4f}  F1={f1:.4f}  MAP@3={m3:.4f}')
    return {'accuracy': acc, 'f1': f1, 'map_at_3': m3}

print(" Helpers ready")

 Helpers ready


In [10]:
raw_train = pd.read_csv(DATA_DIR / 'train.csv')
raw_test  = pd.read_csv(DATA_DIR / 'test.csv')

In [11]:
# Lowercase all text columns
text_cols = ['prompt', 'A', 'B', 'C', 'D', 'E']
for col in text_cols:
    raw_train[col] = raw_train[col].str.lower().str.strip()
    raw_test[col]  = raw_test[col].str.lower().str.strip()

print(f'Train: {raw_train.shape} | Test: {raw_test.shape}')

Train: (2000, 8) | Test: (500, 7)


In [13]:
raw_train.head(1)

,id,prompt,A,B,C,D,E,answer
0,1,pick the best possible answer: what is martin ...,martin heidegger believes that humans exist wi...,martin heidegger believes that humans do not e...,martin heidegger does not believe in the exist...,martin heidegger believes that the relationshi...,martin heidegger believes that time is an illu...,B


In [15]:
# Stratified split before building the combined_text
np.random.seed(42)
train_idx, val_idx = [], []
for ans in 'ABCDE':
    idx = raw_train[raw_train['answer'] == ans].index.tolist()
    np.random.shuffle(idx)
    cut = int(len(idx) * 0.8)
    train_idx += idx[:cut]
    val_idx   += idx[cut:]

train_df = raw_train.loc[train_idx].reset_index(drop=True)
val_df   = raw_train.loc[val_idx].reset_index(drop=True)
test_df  = raw_test.copy()

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

print("train/val split done")

Train: 1599 | Val: 401 | Test: 500
train/val split done
